In [13]:
import geneformer
import inspect

print("=" * 70)
print("GENEFORMER VERSION / TOKENIZER CHECK")
print("=" * 70)

print("Geneformer location:")
print(geneformer.__file__)

print("\nGeneformer version:")
print(getattr(geneformer, "__version__", "version not exposed"))

from geneformer import TranscriptomeTokenizer

print("\nTranscriptomeTokenizer:")
print(TranscriptomeTokenizer)

print("\nConstructor:")
print(inspect.signature(TranscriptomeTokenizer))

GENEFORMER VERSION / TOKENIZER CHECK
Geneformer location:
F:\GeneformerProject\Geneformer\geneformer\__init__.py

Geneformer version:
version not exposed

TranscriptomeTokenizer:
<class 'geneformer.tokenizer.TranscriptomeTokenizer'>

Constructor:
(custom_attr_name_dict=None, nproc=1, chunk_size=512, model_input_size=4096, special_token=True, collapse_gene_ids=True, use_h5ad_index=False, keep_counts=False, model_version='V2', gene_median_file=WindowsPath('F:/GeneformerProject/Geneformer/geneformer/gene_median_dictionary_gc104M.pkl'), token_dictionary_file=WindowsPath('F:/GeneformerProject/Geneformer/geneformer/token_dictionary_gc104M.pkl'), gene_mapping_file=WindowsPath('F:/GeneformerProject/Geneformer/geneformer/ensembl_mapping_dict_gc104M.pkl'))


In [14]:
# Step A1 — inspect the tokenizer's actual tokenization method
from geneformer import TranscriptomeTokenizer

print("=" * 70)
print("GENEFORMER TOKENIZER METHODS")
print("=" * 70)

print("\nTokenize data:")
print(inspect.signature(TranscriptomeTokenizer.tokenize_data))

print("\nTokenizer source location:")
print(inspect.getfile(TranscriptomeTokenizer))


tokenizer = TranscriptomeTokenizer(
    nproc=1,
    model_input_size=4096,
    special_token=True,
    collapse_gene_ids=True,
    use_h5ad_index=False,
    keep_counts=False,
    model_version="V2",
)

print("Tokenizer created successfully.")
print(tokenizer)

GENEFORMER TOKENIZER METHODS

Tokenize data:
(self, data_directory: 'Path | str', output_directory: 'Path | str', output_prefix: 'str', file_format: "Literal['loom', 'h5ad', 'zarr']" = 'loom', input_identifier: 'str' = '', use_generator: 'bool' = False)

Tokenizer source location:
F:\GeneformerProject\Geneformer\geneformer\tokenizer.py
Tokenizer created successfully.


In [15]:
# ============================================================
# STEP B — TCGA TOKENIZATION
# Save directly as Hugging Face Dataset
# ============================================================

import pandas as pd
import numpy as np
import pickle

from pathlib import Path
from tqdm.auto import tqdm

from datasets import Dataset, DatasetDict, ClassLabel


# ============================================================
# 1. PATHS
# ============================================================

BASE_DIR = Path(
    r"F:\GeneformerProject\data\geneformer_ready"
)

OUTPUT_DIR = Path(
    r"F:\GeneformerProject\data\geneformer_tokenized"
)

OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True
)

TOKEN_FILE = Path(
    r"F:\GeneformerProject\Geneformer\geneformer"
    r"\token_dictionary_gc104M.pkl"
)


# ============================================================
# 2. PARAMETERS
# ============================================================

MAX_LENGTH = 4096

# 4096 = CLS + 4094 genes + EOS
N_GENES = MAX_LENGTH - 2

LABEL_COLUMNS = [
    "Cancer_Type",
    "tissue_type"
]

TRAIN_FILE = BASE_DIR / "train_geneformer.parquet"
VAL_FILE   = BASE_DIR / "validation_geneformer.parquet"
TEST_FILE  = BASE_DIR / "test_geneformer.parquet"


# ============================================================
# 3. LOAD GENEFORMER VOCABULARY
# ============================================================

with open(TOKEN_FILE, "rb") as f:
    token_dictionary = pickle.load(f)

CLS_TOKEN = token_dictionary["<cls>"]
EOS_TOKEN = token_dictionary["<eos>"]

print("=" * 70)
print("STEP B — TCGA GENEFORMER TOKENIZATION")
print("=" * 70)

print("Geneformer vocabulary size:", len(token_dictionary))
print("CLS token:", CLS_TOKEN)
print("EOS token:", EOS_TOKEN)
print("Maximum sequence length:", MAX_LENGTH)
print("Genes per sequence:", N_GENES)


# ============================================================
# 4. LOAD ALL THREE DATASETS
# ============================================================

train_df = pd.read_parquet(TRAIN_FILE)
val_df   = pd.read_parquet(VAL_FILE)
test_df  = pd.read_parquet(TEST_FILE)

print("\nDataset sizes:")
print("Train:", train_df.shape)
print("Validation:", val_df.shape)
print("Test:", test_df.shape)


# ============================================================
# 5. IDENTIFY GENES
# ============================================================

gene_columns = [
    c for c in train_df.columns
    if c not in LABEL_COLUMNS
]

print("\nNumber of genes:", len(gene_columns))

assert len(gene_columns) == 15100, (
    f"Expected 15100 genes, found {len(gene_columns)}"
)

# Make sure validation/test have exactly the same genes
assert gene_columns == [
    c for c in val_df.columns
    if c not in LABEL_COLUMNS
]

assert gene_columns == [
    c for c in test_df.columns
    if c not in LABEL_COLUMNS
]


# ============================================================
# 6. CHECK GENEFORMER VOCABULARY
# ============================================================

missing_genes = [
    gene for gene in gene_columns
    if gene not in token_dictionary
]

print("Genes missing from Geneformer vocabulary:",
      len(missing_genes))

assert len(missing_genes) == 0, (
    f"{len(missing_genes)} genes are missing "
    "from Geneformer vocabulary."
)


# ============================================================
# 7. PRECOMPUTE TOKEN IDS
# ============================================================

gene_token_ids = np.array(
    [
        token_dictionary[gene]
        for gene in gene_columns
    ],
    dtype=np.int32
)

print("Gene token array shape:",
      gene_token_ids.shape)


# ============================================================
# 8. CREATE CONSISTENT CANCER LABEL MAPPING
# ============================================================

# Use ONLY training data to define the class mapping.
# This prevents information from validation/test leaking
# into label encoding.

cancer_classes = sorted(
    train_df["Cancer_Type"]
    .dropna()
    .unique()
    .tolist()
)

label_to_id = {
    label: idx
    for idx, label in enumerate(cancer_classes)
}

id_to_label = {
    idx: label
    for label, idx in label_to_id.items()
}

print("\nNumber of cancer classes:",
      len(cancer_classes))

print("\nCancer class mapping:")

for idx, label in id_to_label.items():
    print(f"{idx:3d} -> {label}")


# ============================================================
# 9. CHECK VALIDATION / TEST LABELS
# ============================================================

unknown_val_labels = sorted(
    set(val_df["Cancer_Type"]) - set(label_to_id)
)

unknown_test_labels = sorted(
    set(test_df["Cancer_Type"]) - set(label_to_id)
)

assert len(unknown_val_labels) == 0, (
    "Validation contains cancer classes "
    "not present in training."
)

assert len(unknown_test_labels) == 0, (
    "Test contains cancer classes "
    "not present in training."
)


# ============================================================
# 10. TOKENIZATION FUNCTION
# ============================================================

def tokenize_dataframe(df, dataset_name):

    print("\n" + "=" * 70)
    print("Tokenizing:", dataset_name)
    print("=" * 70)

    X = df[gene_columns].to_numpy(
        dtype=np.float32
    )

    input_ids = []

    labels = []

    cancer_types = []
    tissue_types = []

    for i in tqdm(
        range(X.shape[0]),
        desc=dataset_name
    ):

        values = X[i]

        # ----------------------------------------------------
        # Check that all 15100 values are valid
        # ----------------------------------------------------

        valid = np.isfinite(values)

        if not np.all(valid):
            raise ValueError(
                f"{dataset_name}, sample {i}: "
                "contains NaN or infinite values."
            )

        # ----------------------------------------------------
        # Rank genes by expression
        # Highest expression first
        # ----------------------------------------------------

        order = np.argsort(
            -values,
            kind="stable"
        )

        # ----------------------------------------------------
        # Select top N genes
        # ----------------------------------------------------

        top_indices = order[:N_GENES]

        ranked_tokens = gene_token_ids[top_indices]

        # ----------------------------------------------------
        # Safety check
        # ----------------------------------------------------

        assert len(ranked_tokens) == N_GENES

        # ----------------------------------------------------
        # CLS + 4094 genes + EOS
        # ----------------------------------------------------

        sequence = np.concatenate(
            [
                np.array(
                    [CLS_TOKEN],
                    dtype=np.int32
                ),

                ranked_tokens.astype(
                    np.int32
                ),

                np.array(
                    [EOS_TOKEN],
                    dtype=np.int32
                )
            ]
        )

        # ----------------------------------------------------
        # Final safety checks
        # ----------------------------------------------------

        assert len(sequence) == MAX_LENGTH
        assert sequence[0] == CLS_TOKEN
        assert sequence[-1] == EOS_TOKEN

        input_ids.append(
            sequence.tolist()
        )

        # ----------------------------------------------------
        # Cancer label
        # ----------------------------------------------------

        cancer_type = df.iloc[i]["Cancer_Type"]

        labels.append(
            label_to_id[cancer_type]
        )

        cancer_types.append(
            cancer_type
        )

        tissue_types.append(
            df.iloc[i]["tissue_type"]
        )

    # --------------------------------------------------------
    # Create Hugging Face Dataset
    # --------------------------------------------------------

    dataset_dict = {
        "input_ids": input_ids,
        "labels": labels,
        "Cancer_Type": cancer_types,
        "tissue_type": tissue_types
    }

    dataset = Dataset.from_dict(
        dataset_dict
    )

    return dataset


# ============================================================
# 11. TOKENIZE TRAIN
# ============================================================

train_dataset = tokenize_dataframe(
    train_df,
    "TRAIN"
)


# ============================================================
# 12. TOKENIZE VALIDATION
# ============================================================

validation_dataset = tokenize_dataframe(
    val_df,
    "VALIDATION"
)


# ============================================================
# 13. TOKENIZE TEST
# ============================================================

test_dataset = tokenize_dataframe(
    test_df,
    "TEST"
)


# ============================================================
# 14. CREATE DATASET DICT
# ============================================================

tokenized_dataset = DatasetDict({
    "train": train_dataset,
    "validation": validation_dataset,
    "test": test_dataset
})


# ============================================================
# 15. SAVE HUGGING FACE DATASET
# ============================================================

FINAL_OUTPUT = (
    OUTPUT_DIR / "tcga_geneformer_ranked"
)

tokenized_dataset.save_to_disk(
    str(FINAL_OUTPUT)
)


# ============================================================
# 16. VERIFICATION
# ============================================================

print("\n" + "=" * 70)
print("TOKENIZATION COMPLETE")
print("=" * 70)

print(tokenized_dataset)

print("\nDataset sizes:")

print(
    "Train:",
    len(tokenized_dataset["train"])
)

print(
    "Validation:",
    len(tokenized_dataset["validation"])
)

print(
    "Test:",
    len(tokenized_dataset["test"])
)


# ============================================================
# 17. CHECK ONE SAMPLE
# ============================================================

sample = tokenized_dataset["train"][0]

print("\n" + "=" * 70)
print("FIRST TRAINING SAMPLE")
print("=" * 70)

print("Cancer type:",
      sample["Cancer_Type"])

print("Cancer label:",
      sample["labels"])

print("Sequence length:",
      len(sample["input_ids"]))

print("First 10 tokens:")
print(sample["input_ids"][:10])

print("Last 10 tokens:")
print(sample["input_ids"][-10:])


# ============================================================
# 18. FINAL ASSERTIONS
# ============================================================

assert len(sample["input_ids"]) == 4096
assert sample["input_ids"][0] == CLS_TOKEN
assert sample["input_ids"][-1] == EOS_TOKEN

print("\nFinal checks:")
print("✓ Sequence length = 4096")
print("✓ CLS = 2")
print("✓ EOS = 3")
print("✓ No padding required")
print("✓ Cancer labels encoded consistently")
print("✓ Train/validation/test saved")
print("✓ Hugging Face Dataset format")

STEP E — TCGA GENEFORMER TOKENIZATION
Geneformer vocabulary size: 20275
CLS token: 2
EOS token: 3
Maximum sequence length: 4096
Genes per sequence: 4094

Dataset sizes:
Train: (7742, 15102)
Validation: (1659, 15102)
Test: (1659, 15102)

Number of genes: 15100
Genes missing from Geneformer vocabulary: 0
Gene token array shape: (15100,)

Number of cancer classes: 33

Cancer class mapping:
  0 -> acute myeloid leukemia
  1 -> adrenocortical cancer
  2 -> bladder urothelial carcinoma
  3 -> brain lower grade glioma
  4 -> breast invasive carcinoma
  5 -> cervical & endocervical cancer
  6 -> cholangiocarcinoma
  7 -> colon adenocarcinoma
  8 -> diffuse large B-cell lymphoma
  9 -> esophageal carcinoma
 10 -> glioblastoma multiforme
 11 -> head & neck squamous cell carcinoma
 12 -> kidney chromophobe
 13 -> kidney clear cell carcinoma
 14 -> kidney papillary cell carcinoma
 15 -> liver hepatocellular carcinoma
 16 -> lung adenocarcinoma
 17 -> lung squamous cell carcinoma
 18 -> mesotheliom

TRAIN: 100%|██████████| 7742/7742 [01:42<00:00, 75.83it/s] 



Tokenizing: VALIDATION


VALIDATION: 100%|██████████| 1659/1659 [00:22<00:00, 73.47it/s]



Tokenizing: TEST


TEST: 100%|██████████| 1659/1659 [00:18<00:00, 89.09it/s] 



TOKENIZATION COMPLETE
DatasetDict({
    train: Dataset({
        features: ['input_ids', 'labels', 'Cancer_Type', 'tissue_type'],
        num_rows: 7742
    })
    validation: Dataset({
        features: ['input_ids', 'labels', 'Cancer_Type', 'tissue_type'],
        num_rows: 1659
    })
    test: Dataset({
        features: ['input_ids', 'labels', 'Cancer_Type', 'tissue_type'],
        num_rows: 1659
    })
})

Dataset sizes:
Train: 7742
Validation: 1659
Test: 1659

FIRST TRAINING SAMPLE
Cancer type: uterine corpus endometrioid carcinoma
Cancer label: 31
Sequence length: 4096
First 10 tokens:
[2, 6622, 3880, 5060, 1312, 9847, 395, 16621, 7435, 3558]
Last 10 tokens:
[292, 1686, 5852, 13920, 17632, 5993, 10215, 9307, 4200, 3]

Final checks:
✓ Sequence length = 4096
✓ CLS = 2
✓ EOS = 3
✓ No padding required
✓ Cancer labels encoded consistently
✓ Train/validation/test saved
✓ Hugging Face Dataset format
